In [26]:
import pandas as pd
import numpy as np

In [27]:
dados = r"C:\Users\alex.justino\Documents\CARAMELOprojetoGrupo20.csv" 
df = pd.read_csv(dados, sep= ";", encoding= "latin1")
df


,ID Venda,Loja,ID Loja,Estado Loja,Cidade Loja,Nome Cliente,ID Cliente,Sexo,Idade,Estado Cliente,Cidade Cliente,ID Produto,Categoria Produto,Marca,Data da Compra,Data da Entrega,Metodo de Pagamento,Produto,Valor de Venda
0,1,Melo,49,BA,Salvador,Lucas Silveira,35727,Masculino,24,BA,VitÃ³ria da Conquista,946,EletrÃ´nicos,Lenovo,2024-12-20,2024-12-30,Pix,Drone,"8979,56"
1,10,MendonÃ§a Machado Ltda.,7,SP,Sorocaba,Isabella Campos,2188,Feminino,41,MG,UberlÃ¢ndia,112,EletrÃ´nicos,Apple,2023-11-22,2023-11-28,Dinheiro,Celular,"112146,00"
2,1000,Cunha Mendes S.A.,97,BA,CamaÃ§ari,Marcela Barbosa,2293,Feminino,47,SP,Sorocaba,283,Alimentos,Del Valle,2023-08-25,2023-08-27,Dinheiro,Suco,"37,08"
3,100001,Nunes,77,MG,Contagem,Ravy Cunha,38559,Feminino,59,RJ,PetrÃ³polis,805,EletrÃ´nicos,Dell,2024-08-03,2024-08-04,Dinheiro,Notebook,"13455,14"
4,100007,Cunha Mendes S.A.,97,BA,CamaÃ§ari,Sra. Ana CecÃ­lia Pinto,19274,Masculino,27,SP,SÃ£o Paulo,362,Alimentos,Coca-Cola,2023-02-28,2023-03-08,Pix,Refrigerante,"17,33"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149995,99992,Barbosa,96,RS,Pelotas,Caleb da Mota,223,Masculino,63,BA,Salvador,623,Alimentos,PilÃ£o,2024-11-11,2024-11-15,CartÃ£o,CafÃ©,"33,27"
149996,99994,Aparecida da ConceiÃ§Ã£o e Filhos,15,RJ,Campos dos Goytacazes,Eduardo Cavalcanti,8975,Masculino,39,BA,Salvador,253,Alimentos,Vigorelli,2025-01-11,2025-01-13,Pix,FeijÃ£o,"15,50"
149997,99997,Moreira,53,RJ,Rio de Janeiro,Pedro Miguel Mendes,36587,Masculino,43,RS,Pelotas,535,Alimentos,Camil,2023-09-09,2023-09-15,Dinheiro,Arroz,"27,83"
149998,99998,Andrade,94,SP,Santos,Ana Luiza Porto,3938,Feminino,36,BA,Feira de Santana,183,Alimentos,Barilla,2025-01-01,2025-01-02,Pix,MacarrÃ£o,"61,16"


In [28]:
df["Valor de Venda"] = (
    df["Valor de Venda"]
    .astype(str)
    .str.replace("R$", "", regex=False)
    .str.replace(".", "", regex=False)   # remove milhar
    .str.replace(",", ".", regex=False)  # troca vírgula por ponto
    .str.strip()
)

df["Valor de Venda"] = pd.to_numeric(df["Valor de Venda"], errors="coerce")

In [29]:
df_valor_venda_loja = df.groupby("Loja")["Valor de Venda"].sum().reset_index()
df_valor_venda_loja = df_valor_venda_loja.sort_values(by="Valor de Venda", ascending=False)
df_valor_venda_loja

,Loja,Valor de Venda
10,MendonÃ§a Machado Ltda.,73919670.56
7,Gomes,39371215.05
6,Fernandes,37107099.24
0,Andrade,36868599.90
3,Cunha,36133723.77
12,Moura - ME,35509673.51
11,Moreira,26847044.73
17,Sousa,18350601.22
15,Sampaio Viana Ltda.,18017479.95
19,Vargas,17769332.66


In [32]:
def detectar_outliers(grupo):
    q1 = grupo["Valor de Venda"].quantile(0.25)
    q3 = grupo["Valor de Venda"].quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    grupo["Tipo_Outlier"] = "Normal"

    grupo.loc[grupo["Valor de Venda"] < limite_inferior, "Tipo_Outlier"] = "Baixo"
    grupo.loc[grupo["Valor de Venda"] > limite_superior, "Tipo_Outlier"] = "Alto"

    return grupo

df_outliers = df.groupby("Loja", group_keys=False).apply(detectar_outliers)

# Só os outliers
outliers = df_outliers[df_outliers["Tipo_Outlier"] != "Normal"]

outliers

,ID Venda,ID Loja,Estado Loja,Cidade Loja,Nome Cliente,ID Cliente,Sexo,Idade,Estado Cliente,Cidade Cliente,ID Produto,Categoria Produto,Marca,Data da Compra,Data da Entrega,Metodo de Pagamento,Produto,Valor de Venda,Tipo_Outlier
0,1,49,BA,Salvador,Lucas Silveira,35727,Masculino,24,BA,VitÃ³ria da Conquista,946,EletrÃ´nicos,Lenovo,2024-12-20,2024-12-30,Pix,Drone,8979.56,Alto
1,10,7,SP,Sorocaba,Isabella Campos,2188,Feminino,41,MG,UberlÃ¢ndia,112,EletrÃ´nicos,Apple,2023-11-22,2023-11-28,Dinheiro,Celular,112146.00,Alto
3,100001,77,MG,Contagem,Ravy Cunha,38559,Feminino,59,RJ,PetrÃ³polis,805,EletrÃ´nicos,Dell,2024-08-03,2024-08-04,Dinheiro,Notebook,13455.14,Alto
8,100014,53,RJ,Rio de Janeiro,Dr. Bruno Nunes,37283,Feminino,59,SP,Santos,615,EletrÃ´nicos,Sony,2025-06-08,2025-06-10,Pix,Monitor,15428.64,Alto
13,100021,49,BA,Salvador,Anna Liz Cavalcanti,34781,Feminino,22,RJ,Rio de Janeiro,472,MÃ³veis,Casas Bahia,2023-07-15,2023-07-20,CartÃ£o,SofÃ¡,9267.18,Alto
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149969,99938,49,RS,Novo Hamburgo,Benicio Moura,15183,Masculino,69,SP,SÃ£o Paulo,449,MÃ³veis,Etna,2025-01-19,2025-01-20,Pix,Estante,2321.47,Alto
149974,99947,11,RJ,Rio de Janeiro,CauÃª Machado,28068,Masculino,46,BA,CamaÃ§ari,208,EletrÃ´nicos,Sony,2024-02-17,2024-02-20,Pix,Console de Videogame,10083.16,Alto
149980,99960,82,RJ,Campos dos Goytacazes,Marcela Lima,17109,Feminino,24,BA,Salvador,805,EletrÃ´nicos,Dell,2025-07-07,2025-07-12,CartÃ£o,Notebook,8261.12,Alto
149984,99966,94,SP,Santos,Maria LuÃ­sa Pinto,26870,Feminino,33,BA,CamaÃ§ari,913,MÃ³veis,Magazine Luiza,2024-12-11,2024-12-12,CartÃ£o,Guarda-roupa,26295.92,Alto


In [33]:
# Média geral
media_geral = df["Valor de Venda"].mean()

# Média por produto
media_produto = df.groupby("Produto")["Valor de Venda"].mean().reset_index()

# Diferença em relação à média geral
media_produto["impacto"] = media_produto["Valor de Venda"] - media_geral

# Ordenar pelos que mais impactam
impacto_produtos = media_produto.sort_values(by="impacto", ascending=False)

impacto_produtos

,Produto,Valor de Venda,impacto
48,Teclado,26610.779964,23306.791557
12,Celular,25810.380533,22506.392125
20,Fone de Ouvido,25473.801460,22169.813053
31,Notebook,17583.843997,14279.855589
30,Mouse,17376.067714,14072.079306
47,Tablet,16994.416687,13690.428280
29,Monitor,16210.328211,12906.339803
14,Console de Videogame,15895.259880,12591.271472
43,Smartwatch,14360.498633,11056.510225
46,TV,14286.291695,10982.303288


In [57]:
def detectar_outliers(grupo):
    q1 = grupo["Valor de Venda"].quantile(0.25)
    q3 = grupo["Valor de Venda"].quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    grupo["Tipo_Outlier"] = "Baixo"

    grupo.loc[grupo["Valor de Venda"] < limite_inferior, "Tipo_Outlier"] = "Baixo"
    grupo.loc[grupo["Valor de Venda"] > limite_superior, "Tipo_Outlier"] = "Alto"

    return grupo

df_outliers = df.groupby("Loja", group_keys=False).apply(detectar_outliers)

# Só os outliers
outliers_baixos = df_outliers[df_outliers["Tipo_Outlier"] == "Baixo"]

outliers_baixos


,ID Venda,ID Loja,Estado Loja,Cidade Loja,Nome Cliente,ID Cliente,Sexo,Idade,Estado Cliente,Cidade Cliente,ID Produto,Categoria Produto,Marca,Data da Compra,Data da Entrega,Metodo de Pagamento,Produto,Valor de Venda,Tipo_Outlier
2,1000,97,BA,CamaÃ§ari,Marcela Barbosa,2293,Feminino,47,SP,Sorocaba,283,Alimentos,Del Valle,2023-08-25,2023-08-27,Dinheiro,Suco,37.08,Baixo
4,100007,97,BA,CamaÃ§ari,Sra. Ana CecÃ­lia Pinto,19274,Masculino,27,SP,SÃ£o Paulo,362,Alimentos,Coca-Cola,2023-02-28,2023-03-08,Pix,Refrigerante,17.33,Baixo
5,100010,49,BA,Salvador,Sr. Rhavi Siqueira,237,Feminino,55,BA,Salvador,183,Alimentos,Barilla,2023-04-13,2023-04-14,Pix,MacarrÃ£o,20.33,Baixo
6,100011,40,RS,Caxias do Sul,Isaque Pimenta,25565,Masculino,67,RJ,Rio de Janeiro,283,Alimentos,Del Valle,2025-11-15,2025-11-25,CartÃ£o,Suco,29.87,Baixo
7,100012,53,RJ,Rio de Janeiro,Sr. Gustavo Cavalcanti,27717,Masculino,62,BA,Salvador,286,Livros,Amazon,2025-07-07,2025-07-09,CartÃ£o,A RevoluÃ§Ã£o dos Bichos,157.62,Baixo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149995,99992,96,RS,Pelotas,Caleb da Mota,223,Masculino,63,BA,Salvador,623,Alimentos,PilÃ£o,2024-11-11,2024-11-15,CartÃ£o,CafÃ©,33.27,Baixo
149996,99994,15,RJ,Campos dos Goytacazes,Eduardo Cavalcanti,8975,Masculino,39,BA,Salvador,253,Alimentos,Vigorelli,2025-01-11,2025-01-13,Pix,FeijÃ£o,15.50,Baixo
149997,99997,53,RJ,Rio de Janeiro,Pedro Miguel Mendes,36587,Masculino,43,RS,Pelotas,535,Alimentos,Camil,2023-09-09,2023-09-15,Dinheiro,Arroz,27.83,Baixo
149998,99998,94,SP,Santos,Ana Luiza Porto,3938,Feminino,36,BA,Feira de Santana,183,Alimentos,Barilla,2025-01-01,2025-01-02,Pix,MacarrÃ£o,61.16,Baixo


In [47]:
df_outliers["Tipo_Outlier"].value_counts()

Tipo_Outlier
Normal    114481
Alto       35519
Name: count, dtype: int64

In [53]:
def detectar_outliers(grupo):
    grupo = grupo.copy()

    q1 = grupo["Valor de Venda"].quantile(0.25)
    q3 = grupo["Valor de Venda"].quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    grupo["Tipo_Outlier"] = "Normal"

    grupo.loc[grupo["Valor de Venda"] < limite_inferior, "Tipo_Outlier"] = "Baixo"
    grupo.loc[grupo["Valor de Venda"] > limite_superior, "Tipo_Outlier"] = "Alto"

    return grupo

In [54]:
df_outliers_loja = df.groupby("Loja", group_keys=False).apply(detectar_outliers)

outliers_loja = df_outliers_loja[df_outliers_loja["Tipo_Outlier"] != "Normal"]

In [66]:
df_outliers = df.groupby("Loja", group_keys=False).apply(detectar_outliers)

In [59]:
outliers_baixos = df_outliers[df_outliers["Tipo_Outlier"] == "Baixo"]

In [60]:
outliers_baixos_ordenados = outliers_baixos.sort_values(
    by="Valor de Venda", ascending=True
)

In [61]:
top10_baixos = outliers_baixos_ordenados.head(10)

top10_baixos

,ID Venda,ID Loja,Estado Loja,Cidade Loja,Nome Cliente,ID Cliente,Sexo,Idade,Estado Cliente,Cidade Cliente,ID Produto,Categoria Produto,Marca,Data da Compra,Data da Entrega,Metodo de Pagamento,Produto,Valor de Venda,Tipo_Outlier
6861,11225,50,RJ,Rio de Janeiro,Sr. Lorenzo Pires,3393,Masculino,69,RS,Pelotas,623,Alimentos,PilÃ£o,2025-05-17,2025-05-21,CartÃ£o,CafÃ©,4.0,Baixo
82184,248105,17,MG,UberlÃ¢ndia,Dr. Matheus Campos,39190,Feminino,68,RS,Pelotas,300,Alimentos,Bauducco,2025-03-28,2025-04-06,CartÃ£o,Biscoito,4.0,Baixo
82007,247783,97,BA,CamaÃ§ari,Amanda Sampaio,39470,Feminino,61,RS,Caxias do Sul,183,Alimentos,Barilla,2023-05-17,2023-05-23,Pix,MacarrÃ£o,4.0,Baixo
102160,284169,96,RS,Pelotas,Evelyn Rezende,28757,Feminino,60,MG,Belo Horizonte,623,Alimentos,PilÃ£o,2024-05-04,2024-05-14,Pix,CafÃ©,4.0,Baixo
9921,117748,49,RS,Novo Hamburgo,Ana Luiza Barros,14883,Feminino,54,SP,SÃ£o Paulo,183,Alimentos,Barilla,2025-03-30,2025-04-07,CartÃ£o,MacarrÃ£o,4.0,Baixo
100124,280570,77,MG,Contagem,Nicole Cirino,6707,Feminino,28,SP,SÃ£o Paulo,362,Alimentos,Coca-Cola,2024-05-15,2024-05-24,CartÃ£o,Refrigerante,4.0,Baixo
59123,20666,11,RJ,Rio de Janeiro,JoÃ£o Pedro FogaÃ§a,15151,Masculino,18,MG,Belo Horizonte,283,Alimentos,Del Valle,2025-05-04,2025-05-05,Pix,Suco,4.0,Baixo
73022,231639,55,MG,Contagem,Sophia Viana,5931,Feminino,31,BA,Salvador,362,Alimentos,Coca-Cola,2023-05-30,2023-06-02,CartÃ£o,Refrigerante,4.0,Baixo
24808,144729,11,RJ,Rio de Janeiro,Pedro Miguel Pacheco,39843,Masculino,32,BA,Salvador,362,Alimentos,Coca-Cola,2025-03-25,2025-03-30,CartÃ£o,Refrigerante,4.0,Baixo
70694,227459,15,RJ,Campos dos Goytacazes,Davi Luiz Pimenta,29080,Masculino,22,BA,Salvador,183,Alimentos,Barilla,2024-05-02,2024-05-06,Pix,MacarrÃ£o,4.0,Baixo


In [63]:
outliers_altos = df_outliers[df_outliers["Tipo_Outlier"] == "Alto"]

In [64]:
outliers_altos_ordenados = outliers_altos.sort_values(
    by="Valor de Venda", ascending=False
)

In [98]:
top10_altos = outliers_altos_ordenados.head(10)

top10_altos

,ID Venda,ID Loja,Estado Loja,Cidade Loja,Nome Cliente,ID Cliente,Sexo,Idade,Estado Cliente,Cidade Cliente,ID Produto,Categoria Produto,Marca,Data da Compra,Data da Entrega,Metodo de Pagamento,Produto,Valor de Venda,Tipo_Outlier
135398,73899,7,SP,Sorocaba,Luiz OtÃ¡vio da Rosa,12248,Masculino,55,RS,Caxias do Sul,3,EletrÃ´nicos,Apple,2024-12-27,2024-12-30,CartÃ£o,Teclado,168863.37,Alto
26644,148067,7,SP,Sorocaba,Ravi Lucca Farias,36904,Masculino,68,BA,Feira de Santana,541,EletrÃ´nicos,Apple,2024-12-06,2024-12-15,Pix,Fone de Ouvido,168744.27,Alto
111536,31111,7,SP,Sorocaba,Calebe Lima,13705,Masculino,36,MG,Belo Horizonte,112,EletrÃ´nicos,Apple,2023-12-27,2023-12-31,Dinheiro,Celular,167663.33,Alto
81377,246613,7,SP,Sorocaba,LÃ©o Duarte,418,Masculino,39,MG,Juiz de Fora,112,EletrÃ´nicos,Apple,2025-11-21,2025-11-22,Pix,Celular,165877.57,Alto
57311,203339,7,SP,Sorocaba,Fernanda Rodrigues,25835,Feminino,51,MG,Betim,3,EletrÃ´nicos,Apple,2025-11-14,2025-11-15,CartÃ£o,Teclado,165617.55,Alto
26113,147135,7,SP,Sorocaba,Ravi Peixoto,4183,Masculino,63,BA,Itabuna,112,EletrÃ´nicos,Apple,2023-11-28,2023-12-08,Dinheiro,Celular,163939.76,Alto
79985,244109,7,SP,Sorocaba,Davi Pacheco,17920,Feminino,66,BA,Salvador,541,EletrÃ´nicos,Apple,2025-11-13,2025-11-19,Pix,Fone de Ouvido,163864.73,Alto
125671,56383,7,SP,Sorocaba,Emanuella da Cunha,27181,Feminino,67,RS,Pelotas,3,EletrÃ´nicos,Apple,2025-12-16,2025-12-23,Dinheiro,Teclado,163407.27,Alto
98475,2776,7,SP,Sorocaba,Lucca AragÃ£o,38579,Masculino,18,MG,UberlÃ¢ndia,541,EletrÃ´nicos,Apple,2023-11-02,2023-11-08,Pix,Fone de Ouvido,162576.44,Alto
28455,151331,7,SP,Sorocaba,Rael Cavalcanti,10304,Masculino,53,SP,SÃ£o Paulo,541,EletrÃ´nicos,Apple,2023-11-26,2023-12-06,Dinheiro,Fone de Ouvido,161509.47,Alto


In [74]:
def top_outliers_por_loja(grupo):
    altos = grupo[grupo["Tipo_Outlier"] == "Alto"].sort_values(
        by="Valor de Venda", ascending=False
    ).head(5)

    baixos = grupo[grupo["Tipo_Outlier"] == "Baixo"].sort_values(
        by="Valor de Venda", ascending=True
    ).head(5)

    return pd.concat([altos, baixos])

In [76]:
top_outliers_loja = df_outliers.groupby("ID Loja", group_keys=False).apply(top_outliers_por_loja)

top_outliers_loja


,ID Venda,Estado Loja,Cidade Loja,Nome Cliente,ID Cliente,Sexo,Idade,Estado Cliente,Cidade Cliente,ID Produto,Categoria Produto,Marca,Data da Compra,Data da Entrega,Metodo de Pagamento,Produto,Valor de Venda,Tipo_Outlier
19337,134868,SP,RibeirÃ£o Preto,Emanuel Almeida,21990,Masculino,67,BA,Salvador,541,EletrÃ´nicos,Apple,2023-07-16,2023-07-17,Dinheiro,Fone de Ouvido,85800.00,Alto
16244,129335,SP,RibeirÃ£o Preto,Arthur da Costa,36907,Masculino,47,BA,Salvador,112,EletrÃ´nicos,Apple,2023-05-15,2023-05-18,CartÃ£o,Celular,75119.50,Alto
126141,57194,SP,RibeirÃ£o Preto,Kaique Moraes,16391,Masculino,62,SP,Santos,541,EletrÃ´nicos,Apple,2024-07-03,2024-07-06,Dinheiro,Fone de Ouvido,73731.20,Alto
78896,242143,SP,RibeirÃ£o Preto,Maya Barros,34739,Feminino,45,RJ,PetrÃ³polis,541,EletrÃ´nicos,Apple,2025-04-14,2025-04-18,Dinheiro,Fone de Ouvido,73671.71,Alto
145773,92386,SP,RibeirÃ£o Preto,Nicolas Fernandes,12382,Masculino,29,RJ,NiterÃ³i,112,EletrÃ´nicos,Apple,2025-12-23,2025-12-26,Pix,Celular,71782.80,Alto
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149808,99650,BA,CamaÃ§ari,Dr. Paulo Macedo,4850,Feminino,38,BA,CamaÃ§ari,300,Alimentos,Bauducco,2025-03-19,2025-03-26,Pix,Biscoito,4.00,Baixo
148302,96969,BA,CamaÃ§ari,Mirella Ferreira,38577,Feminino,21,RS,Porto Alegre,300,Alimentos,Bauducco,2024-05-08,2024-05-18,Pix,Biscoito,4.00,Baixo
1564,102806,BA,CamaÃ§ari,Cecilia Vasconcelos,192,Feminino,48,MG,Belo Horizonte,283,Alimentos,Del Valle,2023-05-15,2023-05-23,Dinheiro,Suco,4.00,Baixo
146404,93528,BA,CamaÃ§ari,Ana Vargas,28175,Feminino,44,SP,SÃ£o Paulo,183,Alimentos,Barilla,2023-05-03,2023-05-05,Dinheiro,MacarrÃ£o,4.00,Baixo
